[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [Pandas, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/pandas-deep-dive.html)

# Inspecting


## What you will be able to do

Look at a table you did not build and say how many rows it has, what each column holds, where values
are missing and what range the numbers cover, before you ask it anything else.

## The idea

### The problem

A colleague sends you a year of station readings as a table. You print it, and you see ten rows, the
first five and the last five, with a note that there are 35,040 in all. Everything in between is out
of sight. You cannot tell from those ten rows whether a station stopped sending for a week, whether a
column of numbers was read as text, or whether one reading of 999 is hiding in the middle of July.

Each of those surprises shows up later as a wrong answer. A mean that includes a 999 is too high. A
column of numbers stored as text cannot be averaged at all. A count of readings that quietly leaves
out the missing ones is a different count from the number of rows. Finding out at that stage means
working back from a strange result to its cause, which takes far longer than looking first.

Reading 35,040 rows yourself is no way to look. You need a handful of questions that pandas answers
about every row at once: how big is the table, what type is each column, how many values are
missing, and what do the numbers range over.

### What inspecting means

> **Inspecting** a DataFrame means asking pandas to describe it instead of reading it: its size with
> `shape`, its columns and their types with `dtypes` and `info`, its missing values with `isna`, and
> a summary of each column's values with `describe`, `value_counts` and `nunique`. Each of these
> reads the whole table, where printing it shows only the first and last rows.

### Why it works that way

Printing a table shows its edges because a table of 35,040 rows does not fit on a screen, and the
first and last rows say little about the middle. The inspecting methods summarize instead. `info`
counts the values in each column that are present, so a column with gaps shows a smaller count than
the number of rows. `describe` gives the count, mean, spread, extremes and quartiles of every column
of numbers or dates, which is where a 999 or an impossible date shows up.

Each summary leaves something out, and the leaving out is where the mistakes come from. `describe`
skips columns of text unless you ask for them. `count` skips missing values, so it can disagree with
`len`. `info` measures a column of text by its pointers and not by the text, and marks the total
with a `+` to say so. The worked examples show each one, and the Common errors section shows what
each one hides.

### Where this shows up

Inspecting belongs in the first cell after any file is loaded. The **Reading Data** notebook loads files with
`read_csv`, and the next thing to run on any of them is `info`. The **Missing Data** notebook starts
from the counts `isna` gives here, and the **groupby** notebook answers the per-station questions that
`describe` raises. The **NumPy, Deep Dive** guide computed means and extremes one function at a time,
and `describe` prints all of them for every column at once.

### What this notebook covers

- the first and last rows with `head` and `tail`, and a random few with `sample`
- the size of the table, its columns and its index
- each column's type with `dtypes`, and the whole table with `info`
- a summary of the numbers and the times with `describe`, overall and for each station
- the text column with `nunique` and `value_counts`
- where values are missing, with `isna`
- four errors, among them a count that disagrees with the number of rows

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet: read it,
and read the output underneath it. Everything from Setup onward is where you start running things.

```python
import pandas as pd

readings = pd.DataFrame({
    "station": ["Bergen", "Oslo", "Bergen", "Oslo"],
    "celsius": [8.2, 6.1, None, 5.8],
})
print(readings.shape)
print(readings["celsius"].describe())
```

```
(4, 2)
count    3.00000
mean     6.70000
std      1.30767
min      5.80000
25%      5.95000
50%      6.10000
75%      7.15000
max      8.20000
Name: celsius, dtype: float64
```

The table has 4 rows and 2 columns, and `describe` counted 3 readings, because one is missing. The
mean, the smallest and largest values and the quartiles came from those 3. A count smaller than the
number of rows is the first sign of a gap.

## Setup

One library, three helpers for building the readings, and two tables.

- `pandas`, imported as `pd`, is the library, and the cell prints its version
- `math`, `datetime` and `timedelta` build the readings
- `StringIO`, imported later, in the section that needs it, catches the report `info` prints, in
  the last of the Common errors

`year_of_readings` makes a reading for every hour of 2025 at the four stations in `STATIONS`, by the
same formula as the **sqlite3, Deep Dive** guide. Svalbard sent nothing on 2 March, so those 24
readings are missing. `readings` is the table of all 35,040, with the columns `station`, `time` and
`celsius`, and `stations` holds each station's latitude, Kirkenes included.

In [1]:
import math
from datetime import datetime, timedelta

import pandas as pd

STATIONS = {"Bergen": 8.0, "Oslo": 6.5, "Svalbard": -4.5, "Tromso": 3.5}   # each station's mean for the year
LATITUDES = {"Bergen": 60.39, "Oslo": 59.91, "Svalbard": 78.22, "Tromso": 69.65, "Kirkenes": 69.73}


def year_of_readings():
    """Every hour of 2025 at the four stations, as the sqlite3 guide makes them, with Svalbard silent on 2 March."""
    for n in range(365 * 24):
        hour = datetime(2025, 1, 1) + timedelta(hours=n)
        season = -math.cos(2 * math.pi * (n - 400) / (365 * 24))
        day = -math.cos(2 * math.pi * (hour.hour - 3) / 24)
        for i, (station, mean) in enumerate(STATIONS.items()):
            if station == "Svalbard" and hour.strftime("%Y-%m-%d") == "2025-03-02":
                celsius = None
            else:
                wobble = ((n * 37 + i * 101) % 17 - 8) / 10
                celsius = round(mean + 9 * season + 3 * day + wobble, 1) + 0.0
            yield station, hour.strftime("%Y-%m-%dT%H:%M"), celsius


readings = pd.DataFrame(list(year_of_readings()), columns=["station", "time", "celsius"])
readings["time"] = pd.to_datetime(readings["time"], format="%Y-%m-%dT%H:%M")
stations = pd.DataFrame({"station": list(LATITUDES), "latitude": list(LATITUDES.values())})

print("pandas", pd.__version__, "|", len(readings), "readings,", len(stations), "stations")

pandas 2.2.3 | 35040 readings, 5 stations


## Worked examples

### The first and last rows

`head` returns the first rows, five unless you say how many, and `tail` the last:

In [2]:
readings.head()

,station,time,celsius
0,Bergen,2025-01-01 00:00:00,-3.6
1,Oslo,2025-01-01 00:00:00,-3.5
2,Svalbard,2025-01-01 00:00:00,-14.6
3,Tromso,2025-01-01 00:00:00,-6.7
4,Bergen,2025-01-01 01:00:00,-3.7


In [3]:
readings.tail(3)

,station,time,celsius
35037,Oslo,2025-12-31 23:00:00,-3.3
35038,Svalbard,2025-12-31 23:00:00,-14.4
35039,Tromso,2025-12-31 23:00:00,-6.5


The first four rows are midnight on 1 January, one for each station, and the fifth is Bergen at
01:00. The last are 23:00 on 31 December. `sample` picks rows at random from anywhere in the table,
which shows more of the middle. `random_state=0` fixes which rows it picks, so the same five come
back on every run:

In [4]:
readings.sample(5, random_state=0)

,station,time,celsius
5063,Tromso,2025-02-22 17:00:00,-1.9
35039,Tromso,2025-12-31 23:00:00,-6.5
22416,Bergen,2025-08-22 12:00:00,18.4
13063,Tromso,2025-05-17 01:00:00,4.3
32985,Oslo,2025-12-10 14:00:00,1.7


### The size of the table

`shape` is the number of rows and the number of columns, and `len` the number of rows alone:

In [5]:
print(readings.shape)
print(len(readings))
print(readings.columns.tolist())
print(readings.index)

(35040, 3)
35040
['station', 'time', 'celsius']
RangeIndex(start=0, stop=35040, step=1)


`shape` is an attribute, so it takes no parentheses. The columns are `station`, `time` and `celsius`,
and the index is a `RangeIndex` from 0 up to 35040, which means the rows are numbered 0 to 35039.

### What each column holds

`dtypes` gives each column's type:

In [6]:
readings.dtypes

station            object
time       datetime64[ns]
celsius           float64
dtype: object

`object` is the type pandas gives a column of Python strings. `datetime64[ns]` is a date and time,
stored to the nanosecond, and `float64` a decimal. `info` puts the types beside the number of values
each column has:

In [7]:
readings.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 35040 entries, 0 to 35039
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype         
---  ------   --------------  -----         
 0   station  35040 non-null  object        
 1   time     35040 non-null  datetime64[ns]
 2   celsius  35016 non-null  float64       
dtypes: datetime64[ns](1), float64(1), object(1)
memory usage: 821.4+ KB


`station` and `time` have 35,040 values each, one for every row. `celsius` has 35,016, so 24
readings are missing. That is the first sign of Svalbard's silent day, and `info` showed it without
anyone looking for it. The last line is the table's size in memory, which the Common errors section
comes back to.

### A summary of the numbers and the times

In [8]:
readings.describe().round(2)

,time,celsius
count,35040,35016.00
mean,2025-07-02 11:30:00,3.38
min,2025-01-01 00:00:00,-17.30
25%,2025-04-02 05:45:00,-2.30
50%,2025-07-02 11:30:00,3.10
75%,2025-10-01 17:15:00,10.10
max,2025-12-31 23:00:00,20.80
std,NaN,8.27


`describe` summarized `time` and `celsius`, the columns of dates and of numbers. For the readings,
`count` is 35,016 again, the mean is about 3.38 degrees, the coldest reading is -17.3 and the
warmest 20.8, and half of them lie between the quartiles, -2.3 and 10.1. For the times, the earliest
and latest are the first and last hours of 2025, as they should be. `describe` computes no standard
deviation for dates, so `std` is `NaN` in the `time` column. `station` is not in the summary, and
the Common errors section explains why.

### A summary for each station

The overall summary mixes four stations with very different climates. `describe` after `groupby`
gives one row for each station:

In [9]:
readings.groupby("station")["celsius"].describe().round(2)

,count,mean,std,min,25%,50%,75%,max
station,,,,,,,,
Bergen,8760.0,8.00,6.73,-4.8,2.1,8.0,13.9,20.8
Oslo,8760.0,6.50,6.73,-6.3,0.6,6.5,12.4,19.3
Svalbard,8736.0,-4.48,6.73,-17.3,-10.4,-4.5,1.4,8.3
Tromso,8760.0,3.50,6.73,-9.2,-2.4,3.5,9.4,16.2


Bergen, Oslo and Tromso have the means the formula gives them. Svalbard's, -4.48, is a little above
its -4.5, because the day it lost was a cold one in early March. The four standard deviations are the
same, 6.73, because every station has the same seasonal and daily swings around its own mean. Svalbard's
count is 8,736, 24 fewer than the others, which places the missing readings at one station. The
**groupby** notebook explains what `groupby` does before `describe` runs.

### The column of text

For text, the useful questions are how many different values there are and how often each one
appears:

In [10]:
print(readings["station"].nunique())
print(readings["station"].unique())
readings["station"].value_counts().sort_index()

4
['Bergen' 'Oslo' 'Svalbard' 'Tromso']


station
Bergen      8760
Oslo        8760
Svalbard    8760
Tromso      8760
Name: count, dtype: int64

Four stations, with 8,760 rows each. `value_counts` sorts from the most common value down, and these
four tie, so `sort_index` puts them in alphabetical order, which does not depend on how the sort
ordered the tie. With the missing readings dropped first, the counts differ:

In [11]:
readings.dropna()["station"].value_counts().sort_index()

station
Bergen      8760
Oslo        8760
Svalbard    8736
Tromso      8760
Name: count, dtype: int64

`value_counts` works on a column of numbers too. Here are the five readings that occur most often:

In [12]:
print(readings["celsius"].nunique(), "different readings")
readings["celsius"].value_counts().head()

382 different readings


celsius
1.0    194
0.8    192
1.5    187
1.4    186
0.4    185
Name: count, dtype: int64

The readings have one decimal place, so the 35,016 of them take only 382 different values, and 1.0
degrees occurs 194 times, more than any other.

### Where values are missing

`isna` marks every missing value with `True`, and `sum` counts the `True` values in each column:

In [13]:
readings.isna().sum()

station     0
time        0
celsius    24
dtype: int64

Only `celsius` has missing values. Keeping the rows where it is missing shows when they were:

In [14]:
silent = readings[readings["celsius"].isna()]

print(silent["station"].unique())
print(silent["time"].dt.strftime("%Y-%m-%d").unique())

['Svalbard']
['2025-03-02']


All 24 belong to Svalbard, on 2 March. The **Missing Data** notebook covers what to do with them.

## Your turn

Five tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than
getting there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/pandas-deep-dive/05-inspecting-solutions.ipynb).

**1.** Print the shape of `stations` and its column types.

In [15]:
# your code here

**2.** Show the last five rows of `readings` with `tail`, and say which day and hours they are.

In [16]:
# your code here

**3.** Use `describe` on Oslo's readings alone, rounded to two places.

In [17]:
# your code here

**4.** Find the reading that occurs most often at Svalbard, with `value_counts`.

In [18]:
# your code here

**5.** Use `describe` on `stations`, and find the station whose latitude is the `max` it reports.

In [19]:
# your code here

## Common errors

### TypeError: 'tuple' object is not callable

In [20]:
readings.shape()

TypeError: 'tuple' object is not callable

`shape` is an attribute that holds a tuple, `(35040, 3)`, and the parentheses tried to call that
tuple as a function. Write `readings.shape`. `dtypes`, `columns` and `index` are attributes too, while
`head`, `info` and `describe` are methods, which take parentheses.

### No error, and describe leaves out the station names

In [21]:
readings.describe().columns.tolist()

['time', 'celsius']

`describe` summarizes numbers and dates by default, and a column of text has no mean or quartiles,
so `station` is left out without a word. Ask for text columns with `include`, or use `nunique` and
`value_counts`, as the worked examples did:

In [22]:
readings.describe(include="object").loc[["count", "unique"]]

,station
count,35040
unique,4


`include="object"` asks for the text columns. Its summary also has rows called `top` and `freq`, the
most common value and how often it appears, which say little here, since all four stations tie.

### No error, and a count of 35,016 where 35,040 was expected

In [23]:
print("rows:", len(readings))
print("count:", readings["celsius"].count())

rows: 35040
count: 35016


`count` counts the values that are present and skips the missing ones, so it answers "how many
readings arrived?" and `len` answers "how many rows are there?" The two differ by Svalbard's 24
missing hours. Use `len` for the size of a table, and `count` when the missing values should not
count.

### No error, and info's memory usage leaves out the text

In [24]:
from io import StringIO

for depth in [True, "deep"]:
    buffer = StringIO()
    readings.info(buf=buffer, memory_usage=depth)
    print(buffer.getvalue().splitlines()[-1])

memory usage: 821.4+ KB
memory usage: 2.4 MB


`info` writes its report to the screen, so `buf=buffer` sends it into a `StringIO` instead, and the
loop prints the last line of each report. The default reports `821.4+ KB`. The `+` meant that the
`station` column was measured by its pointers to the strings and not by the strings themselves.
`memory_usage="deep"` adds the size of the string each row points to, and reports 2.4 MB, about three
times as much. `memory_usage(deep=True)` gives the same measurement for each column:

In [25]:
readings.memory_usage(deep=True)

Index          132
station    1927200
time        280320
celsius     280320
dtype: int64

pandas counts 1,927,200 bytes for `station`, most of the table, a pointer for each of the 35,040
rows and the size of the name it points to. The **Performance and Memory** notebook stores it in a
fraction of that.

## Recap

- Show the first, last and random rows of a table with `head`, `tail` and `sample`
- Read a table's size from `shape` and its column types from `dtypes` and `info`
- Summarize the numbers and dates with `describe`, overall and for each group
- Count a text column's values with `nunique` and `value_counts`, and find missing values with
  `isna`
- Tell `count` from `len`, and read the `+` in `info`'s memory usage

## What is next

The **Selecting Columns** notebook picks out the columns a question needs, and compares the two ways
pandas offers of writing that.

---

&#8592; **Previous:** [Reading Data](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/pandas-deep-dive/04-reading-data.ipynb)  &nbsp;·&nbsp;  [Pandas, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/pandas-deep-dive.html)  &nbsp;·&nbsp;  **Next:** [Selecting Columns](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/pandas-deep-dive/06-selecting-columns.ipynb) &#8594;
